In [ ]:
import os
import math
import pandas as pd
import numpy as np
import MDAnalysis as mda
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def parse_gro(file_path):
    """Parse a .gro file to extract atom coordinates."""
    with open(file_path, 'r') as f:
        lines = f.readlines()
        
    # Skip header, read the number of atoms (2nd line), and atom data
    atom_lines = lines[2:-1]  # Exclude header, number of atoms, and box vectors
    atom_data = []
    
    for line in atom_lines:
        x = float(line[20:28].strip())
        y = float(line[28:36].strip())
        z = float(line[36:44].strip())
        atom_data.append((x, y, z))
        
    return atom_data

def plot_gro(file_path):
    """Plot the atom coordinates in a 3D space."""
    atom_data = parse_gro(file_path)
    x, y, z = zip(*atom_data)  # Separate coordinates for plotting
    
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(x, y, z, s=10, c='b', alpha=0.6)
    
    ax.set_xlabel('X (nm)')
    ax.set_ylabel('Y (nm)')
    ax.set_zlabel('Z (nm)')
    ax.set_title("Visualization of .gro File Structure")
    plt.show()

# Pulling in MD

## Plan of today's practical

1. Build the structure file
2. Set position restraint

Before starting, let's create a new folder where we are going to work.

In [ ]:
!mkdir practical

In [ ]:
%cd practical

## Building the system

At the end of these two sessions, we want to compute the free energy profile of dissociation of NaCl. For that, we presume that the dissociation process is reversible. We also presume that the collective variable that describes best the reaction coordinate of the dissociation process is simply the distance between the two ions.

Consequently, let's make a simulation when we pull Na and Cl from a distance bigger than the cutoff of the non-bonded interactions to a very small distance.

In [ ]:
!mkdir build

In [ ]:
%cd build

Let's have a look at the structure file containing $Na^+$ and $Cl^-$.

In [ ]:
!cat ../../structures/nacl.gro

We can also have a quick look at it:

In [ ]:
# Plot .gro file
plot_gro('../../structures/nacl.gro')

In this initial configuration, the ions are distant by 2 nm.
- Is that enough? Why / Why not?
- Notice the size of the simulation box. What box size would you choose for this simulation?

After answering these questions, let's solvate the box with ions.

In [ ]:
!gmx_mpi solvate -cp ../../structures/nacl.gro -cs spc216.gro -o nacl_solvated.gro

In [ ]:
# Plot .gro file
plot_gro('nacl_solvated.gro')

## Writing input files for GROMACS

Now that we have the structure file with the initial configuration we want, let's create all the other input files necessary to start our simulations:
1. topology file (topol.top)
2. index file (index.ndx)
3. mdp files for energy minimization (em.mdp) and for equilibration (eq.mdp)

### Topology file (topol.top)

Modify the following script to write the correct topol.top file:


In [ ]:
topology = """; force field parameters
#include "../../force_field/charmm36-jul2022.ff/forcefield.itp"
#include "../../force_field/charmm36-jul2022.ff/tip3p.itp"
#include "../../force_field/charmm36-jul2022.ff/ions.itp"

[ system ]
; name of the system
NaCl ion pair in water

[ molecules ]
; compound   #mols
NA           1
CL           1
SOL          4055
"""

with open('topol.top', 'w') as outFile:
    outFile.write(topology)

### Index file (index.ndx)

The index files allows <code>GROMACS</code> to recognize groups of atoms. It may not be necessary in this simple case, but it is mandatory for most of <i>real</i> systems.

You have seen past time how to create an index file using <code>gmx make_ndx</code>. Here, we will use the library 

In [ ]:
# Load the structure file with MDAnalysis. 
# A "system" in MDAnalysis exists in an object called a Universe
u = mda.Universe('nacl_solvated.gro')

# Let's make selections of the groups of interest
# We want:
# 1. the whole system
# 2. water molecules
# 3. sodium ions
# 4. chloride ions
# 5. all ions
sel_system = u.select_atoms("all")
sel_water = u.select_atoms("resname SOL")
sel_na = u.select_atoms("name NA")
sel_cl = u.select_atoms("name CL")
sel_ions = u.select_atoms("name NA CL")

with mda.selections.gromacs.SelectionWriter("index.ndx", mode="w") as ndx:
    ndx.write(sel_system, name = "System")
    ndx.write(sel_water, name = "Water")
    ndx.write(sel_na, name = "NA")
    ndx.write(sel_cl, name = "CL")
    ndx.write(sel_ions, name = "Ions")

### Energy minimization file (em.mdp)

To minimize the system, we will use the mdp file present in <code>../../input_files/em.mdp</code>.

This file is built based on the categories present in the <a href="https://manual.gromacs.org/2022/user-guide/mdp-options.html">MD parameters page</a>. Note that this page is for the version of <code>GROMACS</code> we will use (2022). It is important to check the page of the version you are using as major updates happened in the past years.

In [ ]:
!cat ../../input_files/em.mdp

### Equilibration file (eq.mdp)

To equilibrate the system, we will use the mdp file present in <code>../../input_files/eq.mdp</code>.

For the equilibration, a few parameters are customary:
- the use of Berendsen algorithm for pressure (and sometimes for temperature) coupling. More recently, v-rescale thermostat, and C-rescale barostat are about to become a new standard, replacing Berendsen. There will be more about coupling algorithms in a later course, but you can already note that.
- the use of a small(er) integration time step than for the production run (here $dt = 1$ fs).

In [ ]:
!cat ../../input_files/eq.mdp

## Position restraints

We want to run the energy minimization and equilibration steps while keeping the initial positions of Na$^+$ and Cl$^-$ fixed in the simulations box.

### Modifying the itp file

For that we will define position restraints in the itp file defining the ions, <code>../../force_field/charmm36-jul2022.ff/ions.itp</code>.

Each molecule is defined starting with a <code>[ moleculetype ]</code> directive. Find sodium and chloride ions, and add the <code>[ position_restraints ]</code> directive, with a force constant on x, y, and z of POSRES_ION_K = 1000 kJ mol$^{-1}$ nm$^{-2}$.

Help:
1. <a href="https://manual.gromacs.org/2022/reference-manual/functions/restraints.html">Restraints in <code>GROMACS</code> 2022</a>
2. <a href="https://manual.gromacs.org/2022/reference-manual/topologies/topology-file-formats.html">File formats in <code>GROMACS</code> 2022</a>

The structure of the directive will be the following:
```
#ifdef POSRES_ION_K
[ position_restraints ]
; id    fct   kx           ky           kz
   <EDIT THIS PART>
#endif
```

### Editing the mdp file

You have probably noticed that the mdp files <code>em.mdp</code> and <code>eq.mdp</code> are not functional. You need to replace the pattern <code>\<POS_RES\></code>. Use <code>sed</code> such that:

```Bash
sed "s/\<POS_RES\>/[REPLACE THIS BRACKET]/" ../../input_files/em.mdp > em.mdp
sed "s/\<POS_RES\>/[REPLACE THIS BRACKET]/" ../../input_files/eq.mdp > eq.mdp
```
    
### About the reference structure file
    
Since our initial structure file directly represents the positions that should be restrained, we can use it as the reference structure file later on.

In [ ]:
!sed "s/<POS_RES>/[REPLACE THIS BRACKET]/" ../../input_files/em.mdp > em.mdp
!sed "s/<POS_RES>/[REPLACE THIS BRACKET]/" ../../input_files/eq.mdp > eq.mdp

## Equilibrating the system

Now that all files are present, let's do the two steps to relax our system and avoid steric clashes.

First, let's create a new directory for the energy minimization.

In [ ]:
%cd ..
!mkdir em
%cd em
!pwd

Let's preprocess our system:

In [ ]:
!gmx_mpi grompp -f ../build/em.mdp -c ../build/nacl_solvated.gro -r ../build/nacl_solvated.gro -p ../build/topol.top -n ../build/index.ndx -o em.tpr

Now, let's run the energy minimization:

In [ ]:
!gmx_mpi mdrun -v -deffnm em

Then, let's create a new directory for the equilibration step, and run it inside.

In [ ]:
%cd ..
!mkdir eq
%cd eq
!pwd

Preprocess (<code>grompp</code>) and run (<code>mdrun</code>) the equilibration step.

In [ ]:
!gmx_mpi grompp -f ../build/eq.mdp -c ../em/em.gro -r ../build/nacl_solvated.gro -p ../build/topol.top -n ../build/index.ndx -o eq.tpr -maxwarn 2

In [ ]:
!gmx_mpi mdrun -v -deffnm eq

### Checking our system

We did some equilibration steps, but how could you check whether your system is close(r) to equilibrium?

One way is to check that the box size does not change with time, using the <code>edr</code> file generated during the MD run.

In [ ]:
!echo -e "5\n11" | gmx_mpi energy -f eq.edr -s eq.tpr -o energy.xvg

Let's plot it!

In [ ]:
# Load the xvg file from gmx energy
xvg_file = 'energy.xvg'
data = np.loadtxt(xvg_file, comments=['@', '#'])

# Separate the x and y data for Potential and Box-X
x = data[:, 0]
box_x = data[:, 2]
pot = data[:, 1]

# Labels
x_label = 'Time [ps]'
y_label_pot = 'Potential energy [kJ/mol]'
y_label_box = 'Box dimension [nm]'

# Create a Matplotlib figure with two subplots (one for each dataset)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# Plot the first dataset in the first subplot
ax1.plot(x, box_x, label='Box-X', linestyle='-')
ax1.set_ylabel(y_label_box)
ax1.set_title('GROMACS Energies')
ax1.legend()

# Plot the second dataset in the second subplot
ax2.plot(x, pot, label='Potential', linestyle='-')
ax2.set_xlabel(x_label)
ax2.set_ylabel(y_label_pot)
ax2.legend()

# Adjust the layout to prevent overlap
plt.tight_layout()

# Show the plot
plt.show()

## Steered MD simulation

We will now use <code>GROMACS</code> pulling code to set up a <i>steered MD</i> simulation.

This simulation will use the distance between the two ions as a collective variable.

This code applies a biasing potential centered in a given value of the collective variable (CV), to restrain the system around this value.

On top, it is possible to change linearly the value of the CV with the simulation time. This way, we can impose continuous changes of the system.

This type of setup is used to generate initial configurations of the system in different states.

We are interested in changing the distance between the ions in the range \[0.3; 2.0] nm.

### MDP file

Let's look at he mdp file dedicated for this task:

In [ ]:
!tail -n 23 ../../input_files/smd.mdp

Do you see any changes in this portion of the mdp file, except the addition of the pulling code?

### Running the Steered MD simulation

Let's preprocess and run this simulation in a new directory.

In [ ]:
%cd ..
!mkdir smd
%cd smd
!pwd

In [ ]:
!gmx_mpi grompp -f ../../input_files/smd.mdp -c ../eq/eq.gro -t ../eq/eq.cpt -p ../build/topol.top -n ../build/index.ndx -o smd.tpr

Note that <code>grompp</code> shows you important information by comparing the position of the center of the biasing potential to the value of the CV that will biased:

```
Pull group  natoms  pbc atom  distance at start  reference at t=0
       1         1         0
       2         1         0       2.001 nm          2.000 nm
```

If those two values are too far from one another, the resulting force from the biasing potential may be too important and make the simulation crash.

In [ ]:
!gmx_mpi mdrun -v -deffnm smd

### Analyzing the output

Two output files linked to the Steered MD procedure are generated:
1. <code>smd_pullx.xvg</code>: the file records the measures the value reached by the system along the collective variable,
2. <code>smd_pullf.xvg</code>: the file records the resulting force applied to the system for a give value of the CV.

Let's plot them:

In [ ]:
# Load the xvg files from the SMD simulation
xvg_file_x = 'smd_pullx.xvg'
data_x = np.loadtxt(xvg_file_x, comments=['@', '#'])
# Load the xvg files from the SMD simulation
xvg_file_f = 'smd_pullf.xvg'
data_f = np.loadtxt(xvg_file_f, comments=['@', '#'])

# Separate the x and y data
x_x = data_x[:, 0]
y_x = data_x[:, 1]
x_f = data_f[:, 0]
y_f = data_f[:, 1]

# Labels
x_label = 'Time [ps]'
y_label_x = 'CV [nm]'
y_label_f = 'Force [kJ/mol/nm]'

# Create a Matplotlib figure with two subplots (one for each dataset)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# Plot the first dataset in the first subplot
ax1.plot(x_x, y_x, label='CV', linestyle='-')
ax1.set_ylabel(y_label_x)
ax1.set_title('GROMACS Steered MD outputs')
ax1.legend()

# Plot the second dataset in the second subplot
ax2.plot(x_f, y_f, label='Force', linestyle='-')
ax2.set_xlabel(x_label)
ax2.set_ylabel(y_label_f)
ax2.legend()

# Adjust the layout to prevent overlap
plt.tight_layout()

# Show the plot
plt.show()

## Do you want to practice more?

### Flat-bottom Restraints vs. Flat-bottom Pulling Code
1. Create a box of water molecules of 5x5x5 nm.
2. Use <code>gmx genion</code> to add 150 mM NaCl.
3. Only run an energy minimization, do not run any equilibration.
4. Remove ions that are more than 1 nm from the center of geometry (COG) of water molecules.
4. Create two systems. In system A, apply restraints, in system B, apply the pulling code.
6. The flat-bottom restraints along z should keep the ions in within 1 nm of the COG of the water molecules. Use a force constant of 500 kJ/mol/nm.
7. Use <code>gmx density</code> to compute the distribution of the ions along z.
8. Are these two methods identical? Is it a problem?